# Baseline hyperparameter sweeps (interactive)

Thin wrapper around `baseline.sweeps.run_sweep` — all logic lives in the `.py` modules so this notebook, the CLI, and `run_sweep.slurm` share one implementation. Each cell runs one small grid sequentially and refreshes `baseline/leaderboard.json`.

Edit the grids in `baseline/sweeps/configs/*.json`. Re-running skips configs whose results already exist (pass `force=True` to redo them).

In [ ]:
import importlib
import sys
from pathlib import Path

# Make `baseline`, `datasets`, `utils` importable.
FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from baseline.sweeps import run_sweep, aggregate, brainwear_aggregate
from baseline.sweeps import train_end_to_end_2d  # ensure 2D trainer is imported

# Reload to pick up any edits made since the kernel started
importlib.reload(run_sweep)
importlib.reload(aggregate)
importlib.reload(brainwear_aggregate)

CONFIGS = FYP_ROOT / 'baseline' / 'sweeps' / 'configs'

In [ ]:
# End-to-end BrainWear sweep
# run_sweep.run(str(CONFIGS / 'sweep_brainwear.json'))

In [ ]:
# End-to-end BraTS proxy sweep
# run_sweep.run(str(CONFIGS / 'sweep_brats.json'))

In [ ]:
# Autoencoder classifier fine-tuning sweep (reuses pre-trained encoders)
# run_sweep.run(str(CONFIGS / 'sweep_ae_classifier.json'))

In [ ]:
# End-to-end BrainWear regression sweep
# run_sweep.run(str(CONFIGS / 'sweep_brainwear_regression.json'))

In [ ]:
# End-to-end BraTS regression sweep
# run_sweep.run(str(CONFIGS / 'sweep_brats_regression.json'))     #3 hours on A40

In [ ]:
# AE-classifier regression sweep
# run_sweep.run(str(CONFIGS / 'sweep_ae_classifier_regression.json'))

## PNG sweeps (2D ResNet on 5-channel quantile-slice inputs)

In [ ]:
# End-to-end BrainWear PNG sweep (categorical, 16 configs)
run_sweep.run(str(CONFIGS / 'sweep_brainwear_png.json'), force=True)

In [ ]:
# End-to-end BraTS PNG sweep (categorical, 4 configs)
run_sweep.run(str(CONFIGS / 'sweep_brats_png.json'), force=True)

In [ ]:
# End-to-end BrainWear PNG regression sweep (16 configs)
run_sweep.run(str(CONFIGS / 'sweep_brainwear_png_regression.json'), force=True)

In [ ]:
# End-to-end BraTS PNG regression sweep (4 configs)
run_sweep.run(str(CONFIGS / 'sweep_brats_png_regression.json'), force=True)

### AE-classifier 2D sweeps
Run the autoencoder pre-training sweep first (cell below), then the classifier fine-tuning sweeps.

In [ ]:
# 2D autoencoder pre-training sweep (4 configs: resnet18/50 x 2 lrs)
run_sweep.run(str(CONFIGS / 'sweep_autoencoder_2d.json'), refresh_leaderboard=False)

In [ ]:
# 2D AE-classifier sweep (categorical, 8 configs) — requires encoder_2d/ weights
run_sweep.run(str(CONFIGS / 'sweep_ae_classifier_2d.json'))

In [ ]:
# 2D AE-classifier regression sweep (8 configs) — requires encoder_2d/ weights
run_sweep.run(str(CONFIGS / 'sweep_ae_classifier_2d_regression.json'))

In [ ]:
# Refresh + show the leaderboard
aggregate.print_table(aggregate.build_leaderboard(write=True))

## Outcome score sweeps (best hyperparams fixed, score varies)

Each sweep below uses the **best hyperparameters** found for that pipeline and sweeps over all 27 EORTC outcome scales. Results go into `baseline/brainwear_leaderboard.json` so you can compare which scale is most predictable by each model type.

Re-running skips scales whose results already exist (pass `force=True` to redo them).

In [ ]:
# 3D end-to-end BrainWear outcome sweep (27 configs: best resnet18 hyperparams x all EORTC scales)
# run_sweep.run(str(CONFIGS / 'sweep_brainwear_outcomes.json'), force=True)

In [ ]:
# 3D AE-classifier outcome sweep (27 configs: best resnet50 encoder hyperparams x all EORTC scales)
# run_sweep.run(str(CONFIGS / 'sweep_ae_classifier_outcomes.json'), force=True)

In [ ]:
# 2D end-to-end BrainWear PNG outcome sweep (27 configs: best resnet50 hyperparams x all EORTC scales)
run_sweep.run(str(CONFIGS / 'sweep_brainwear_png_outcomes.json'), force=True)

In [ ]:
# 2D AE-classifier outcome sweep (27 configs: best resnet18 encoder hyperparams x all EORTC scales)
run_sweep.run(str(CONFIGS / 'sweep_ae_classifier_2d_outcomes.json'), force=True)

In [ ]:
# Refresh + show brainwear_leaderboard.json (outcome score comparison across all pipelines)
brainwear_aggregate.print_table(brainwear_aggregate.build_leaderboard(write=True))